# ☁️ DervGen Model Registry & Cloud Storage Initializer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sautille-stroke/workspace/blob/main/notebooks/02_model_and_drive_management.ipynb)

**Core Principle**: **Zero Local Disk Footprint**.
This notebook manages pre-fetching and organizing our core model weights into Google Drive (`/dervgen/model/{usage}/...`).
- ⚡ **Cloud-to-Cloud Pre-fetcher**: Downloads curated SOTA models (SAM 2, DINOv2, DUSt3R, Marigold, Real-ESRGAN) straight into Drive at 50-150 MB/s.
- 📂 **SSOT Model Layout**: Structured by purpose (`foundation`, `3d_recon`, `texture_gen`, `denoise_upscale`, `actor_dynamics`).
- 🚀 **Per-Job Workspace**: Creates isolated execution sandboxes in `/dervgen/workspace/${jobname}`.

## 1. 💾 Mount Google Drive Cloud

In [ ]:
import os

def ensure_workspace():
    WORKSPACE_DIR = "/content/workspace"
    if os.path.exists(WORKSPACE_DIR):
        print('🔄 检测到已存在 /content/workspace，正在自动同步拉取最新代码...')
        !git -C /content/workspace pull --rebase
        return True
    
    token = ""
    # 1. 尝试从 Colab Secrets 读取
    try:
        from google.colab import userdata
        for key in ['GITHUB_TOKEN', 'GH_TOKEN', 'github_token', 'token']:
            try:
                val = userdata.get(key)
                if val:
                    token = val.strip()
                    print(f"✅ 从 Colab Secrets ({key}) 读取到授权 Token")
                    break
            except Exception:
                pass
    except Exception:
        pass
        
    # 2. 尝试从环境变量读取
    if not token:
        token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN') or ''
        
    # 3. 兜底：如果未配置 Secrets，交互式提示输入
    if not token:
        import getpass
        print("ℹ️ 未在 Colab Secrets 中找到 GITHUB_TOKEN。私有仓库克隆需要 Personal Access Token (PAT)。")
        print("💡 提示：也可以在 Colab 左侧 🔑 Secrets 中添加 GITHUB_TOKEN 并开启 Notebook access。")
        token = getpass.getpass("🔑 请粘贴你的 GitHub Token (输入时不显示，按回车确认): ").strip()
        
    repo_url = f"https://{token}@github.com/sautille-stroke/workspace.git" if token else "https://github.com/sautille-stroke/workspace.git"
    print(f"📥 正在克隆 sautille-stroke/workspace 到 {WORKSPACE_DIR}...")
    !git clone {repo_url} {WORKSPACE_DIR}
    return os.path.exists(WORKSPACE_DIR)

# 1. Ensure Workspace repo
ensure_workspace()

# 2. Mount Google Drive with safe fallback
try:
    from google.colab import drive
    drive_path = "/content/drive"
    if not os.path.exists(drive_path):
        drive.mount(drive_path)
        print("✅ Google Drive 挂载成功")
    else:
        print("ℹ️ Google Drive 已经挂载")
except Exception as e:
    print(f"⚠️ Google Drive 挂载遇到问题: {e}")
    print("💡 提示：无需担心，下载器已支持自动回退至 Colab 本地高速 SSD (/content/dervgen)。")

# 3. Initialize cloud directory hierarchy
if os.path.exists("/content/drive/MyDrive"):
    !python3 /content/workspace/colab/drive_manager.py
    !python3 /content/workspace/colab/drive_manager.py list
else:
    print("ℹ️ 当前使用 Colab 本地存储模式 (/content/dervgen)")


## 2. ⚡ Batch Model Pre-fetcher (One-Click Initialization)
Select your desired preset:
- `fast` (~2.6 GB, 30s-1min on Colab): 12 lightweight models for smoke testing and instant experiments
- `P0` (Production Baseline, ~26.5 GB): Core models (adds TRELLIS, Hunyuan3D-2 Paint, Depth Anything V2, IC-Light, F5-TTS, Wan 2.1, SMPLer-X, UniRig)
- `P1` (~15.1 GB): Secondary and extended models (TripoSR, InstantMesh, SAM 2 Large, DINOv2 Large, Marigold, AudioLDM 2, CLIP)
- `all` (~41.6 GB): Complete 36-model suite across all 7 domains


In [ ]:
#@title Model Pre-fetcher Parameters { run: "auto" }
PRESET = "fast" #@param ["fast", "P0", "P1", "all"]
STORAGE = "auto (Drive or Colab SSD)" #@param ["auto (Drive or Colab SSD)", "google_drive", "local_colab_ssd"]
USAGE_FILTER = "all" #@param ["all", "foundation", "3d_recon", "texture_gen", "video_gen", "actor_dynamics", "audio_foley", "denoise_upscale"]
DRY_RUN = False #@param {type:"boolean"}

import os

def ensure_workspace():
    WORKSPACE_DIR = "/content/workspace"
    if os.path.exists(WORKSPACE_DIR):
        print('🔄 检测到已存在 /content/workspace，正在自动同步拉取最新代码...')
        !git -C /content/workspace pull --rebase
        return True
    
    token = ""
    # 1. 尝试从 Colab Secrets 读取
    try:
        from google.colab import userdata
        for key in ['GITHUB_TOKEN', 'GH_TOKEN', 'github_token', 'token']:
            try:
                val = userdata.get(key)
                if val:
                    token = val.strip()
                    print(f"✅ 从 Colab Secrets ({key}) 读取到授权 Token")
                    break
            except Exception:
                pass
    except Exception:
        pass
        
    # 2. 尝试从环境变量读取
    if not token:
        token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN') or ''
        
    # 3. 兜底：如果未配置 Secrets，交互式提示输入
    if not token:
        import getpass
        print("ℹ️ 未在 Colab Secrets 中找到 GITHUB_TOKEN。私有仓库克隆需要 Personal Access Token (PAT)。")
        print("💡 提示：也可以在 Colab 左侧 🔑 Secrets 中添加 GITHUB_TOKEN 并开启 Notebook access。")
        token = getpass.getpass("🔑 请粘贴你的 GitHub Token (输入时不显示，按回车确认): ").strip()
        
    repo_url = f"https://{token}@github.com/sautille-stroke/workspace.git" if token else "https://github.com/sautille-stroke/workspace.git"
    print(f"📥 正在克隆 sautille-stroke/workspace 到 {WORKSPACE_DIR}...")
    !git clone {repo_url} {WORKSPACE_DIR}
    return os.path.exists(WORKSPACE_DIR)

ensure_workspace()

# 自动处理存储目标（Drive 或 本地 SSD 自动回退）
if "local_colab_ssd" in STORAGE:
    os.environ["DERVGEN_DRIVE_ROOT"] = "/content/dervgen"
    print("🚀 使用 Colab 本地高速 SSD 存储 (/content/dervgen/model)")
elif "google_drive" in STORAGE:
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print(f"⚠️ Google Drive 挂载失败: {e}")
else: # auto
    if os.path.exists("/content/drive/MyDrive"):
        print("💾 已挂载 Google Drive，保存至 /content/drive/MyDrive/dervgen/model")
    else:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ Google Drive 挂载成功")
        except Exception as e:
            print(f"ℹ️ Google Drive 凭据未就绪，已自动切换为【Colab 本地高速 SSD】存储模式 (/content/dervgen/model)")
            os.environ["DERVGEN_DRIVE_ROOT"] = "/content/dervgen"

cmd = f"python3 /content/workspace/colab/init_models.py --preset={PRESET}"
if USAGE_FILTER != "all":
    cmd += f" --usage={USAGE_FILTER}"
if DRY_RUN:
    cmd += " --dry-run"

!{cmd}


## 3. 🌐 Custom Single Model Download
Download any custom weights from web URLs or HuggingFace straight into Drive.

In [ ]:
MODEL_USAGE = "foundation" #@param ["foundation", "3d_recon", "texture_gen", "actor_dynamics", "denoise_upscale"]
DOWNLOAD_URL = "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" #@param {type:"string"}
FILENAME = "dinov2_vits14.pth" #@param {type:"string"}

!python3 /content/workspace/colab/drive_manager.py download_model {MODEL_USAGE} "{DOWNLOAD_URL}" "{FILENAME}"

## 4. 🚀 Create Job Workspace (`/dervgen/workspace/${jobname}`)

In [ ]:
JOB_NAME = "sponza_recon_01" #@param {type:"string"}

# Create isolated job workspace on Drive
!python3 /content/workspace/colab/drive_manager.py create_job {JOB_NAME}

# Write job configuration into input/
import yaml
job_dir = f"/content/drive/MyDrive/dervgen/workspace/{JOB_NAME}"
config = {
    "job_name": JOB_NAME,
    "model": "model/3d_recon/dust3r/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth",
    "data": "data/captures/sponza_daylight",
    "export_ply": True
}
with open(f"{job_dir}/input/job_config.yaml", "w") as f:
    yaml.dump(config, f)
print(f"✅ Wrote config to: {job_dir}/input/job_config.yaml")

## 5. 📊 Storage Inventory Summary

In [ ]:
!python3 /content/workspace/colab/drive_manager.py list